<a href="https://colab.research.google.com/github/Betty-jyw/MMA3001-Occupancy-Prediction/blob/main/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Feature Engineering

## Objective

The purpose of this notebook is to transform the processed occupancy time series into suitable input features for predicting building-space occupancy 30 minutes into the future.

The feature engineering process will focus on information that would be available at the time of prediction, including:

- current occupancy
- recent occupancy history
- time of day
- day of week
- recent occupancy patterns

The prediction target is the occupancy state 30 minutes into the future.

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/MMA3001_Project/data/processed/occupancy_processed.csv"

data = pd.read_csv(
    file_path,
    parse_dates=["collecteddate"]
)

data = data.set_index("collecteddate")

print("Dataset shape:", data.shape)

print("\nColumns:")
print(data.columns.tolist())

data.head(10)

Mounted at /content/drive
Dataset shape: (127428, 2)

Columns:
['current_occupancy', 'target_30min']


,current_occupancy,target_30min
collecteddate,,
2023-11-23 19:15:00+00:00,0.0,0.0
2023-11-23 20:50:00+00:00,0.0,0.0
2023-11-23 20:55:00+00:00,0.0,1.0
2023-11-23 21:00:00+00:00,0.0,1.0
2023-11-23 21:05:00+00:00,0.0,0.0
2023-11-23 21:10:00+00:00,0.0,0.0
2023-11-23 21:15:00+00:00,0.0,0.0
2023-11-23 21:20:00+00:00,0.0,1.0
2023-11-23 21:25:00+00:00,1.0,1.0


In [2]:
time_diff = data.index.to_series().diff()

print("Time difference statistics:")
print(time_diff.describe())

print("\nMost common time differences:")
print(time_diff.value_counts().head(10))

Time difference statistics:
count                       127427
mean     0 days 00:09:57.158373029
std      0 days 01:27:56.717516790
min                0 days 00:05:00
25%                0 days 00:05:00
50%                0 days 00:05:00
75%                0 days 00:05:00
max               15 days 20:20:00
Name: collecteddate, dtype: object

Most common time differences:
collecteddate
0 days 00:05:00    122166
0 days 00:10:00      1355
0 days 00:15:00       760
0 days 00:20:00       707
0 days 00:25:00       446
0 days 00:30:00       340
0 days 01:20:00        86
0 days 01:15:00        63
0 days 00:35:00        57
0 days 01:05:00        56
Name: count, dtype: int64


In [3]:
feature_data = data.copy()

for minutes in [5, 10, 15, 30]:
    past_times = feature_data.index - pd.Timedelta(minutes=minutes)

    feature_data[f"lag_{minutes}min"] = (
        data["current_occupancy"]
        .reindex(past_times)
        .to_numpy()
    )

feature_data.head(10)

,current_occupancy,target_30min,lag_5min,lag_10min,lag_15min,lag_30min
collecteddate,,,,,,
2023-11-23 19:15:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
2023-11-23 20:50:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
2023-11-23 20:55:00+00:00,0.0,1.0,0.0,NaN,NaN,NaN
2023-11-23 21:00:00+00:00,0.0,1.0,0.0,0.0,NaN,NaN
2023-11-23 21:05:00+00:00,0.0,0.0,0.0,0.0,0.0,NaN
2023-11-23 21:10:00+00:00,0.0,0.0,0.0,0.0,0.0,NaN
2023-11-23 21:15:00+00:00,0.0,0.0,0.0,0.0,0.0,NaN
2023-11-23 21:20:00+00:00,0.0,1.0,0.0,0.0,0.0,0.0
2023-11-23 21:25:00+00:00,1.0,1.0,0.0,0.0,0.0,0.0


In [4]:
feature_data["hour"] = feature_data.index.hour
feature_data["day_of_week"] = feature_data.index.dayofweek

feature_data[
    ["current_occupancy", "lag_5min", "lag_10min",
     "lag_15min", "lag_30min", "hour",
     "day_of_week", "target_30min"]
].head(10)

,current_occupancy,lag_5min,lag_10min,lag_15min,lag_30min,hour,day_of_week,target_30min
collecteddate,,,,,,,,
2023-11-23 19:15:00+00:00,0.0,NaN,NaN,NaN,NaN,19,3,0.0
2023-11-23 20:50:00+00:00,0.0,NaN,NaN,NaN,NaN,20,3,0.0
2023-11-23 20:55:00+00:00,0.0,0.0,NaN,NaN,NaN,20,3,1.0
2023-11-23 21:00:00+00:00,0.0,0.0,0.0,NaN,NaN,21,3,1.0
2023-11-23 21:05:00+00:00,0.0,0.0,0.0,0.0,NaN,21,3,0.0
2023-11-23 21:10:00+00:00,0.0,0.0,0.0,0.0,NaN,21,3,0.0
2023-11-23 21:15:00+00:00,0.0,0.0,0.0,0.0,NaN,21,3,0.0
2023-11-23 21:20:00+00:00,0.0,0.0,0.0,0.0,0.0,21,3,1.0
2023-11-23 21:25:00+00:00,1.0,0.0,0.0,0.0,0.0,21,3,1.0


In [5]:
lag_columns = [
    "lag_5min",
    "lag_10min",
    "lag_15min",
    "lag_30min"
]

feature_data["recent_occupancy_rate"] = (
    feature_data[
        ["current_occupancy"] + lag_columns
    ].mean(axis=1)
)

feature_data[
    ["current_occupancy", "lag_5min", "lag_10min",
     "lag_15min", "lag_30min",
     "recent_occupancy_rate", "target_30min"]
].head(10)

,current_occupancy,lag_5min,lag_10min,lag_15min,lag_30min,recent_occupancy_rate,target_30min
collecteddate,,,,,,,
2023-11-23 19:15:00+00:00,0.0,NaN,NaN,NaN,NaN,0.0,0.0
2023-11-23 20:50:00+00:00,0.0,NaN,NaN,NaN,NaN,0.0,0.0
2023-11-23 20:55:00+00:00,0.0,0.0,NaN,NaN,NaN,0.0,1.0
2023-11-23 21:00:00+00:00,0.0,0.0,0.0,NaN,NaN,0.0,1.0
2023-11-23 21:05:00+00:00,0.0,0.0,0.0,0.0,NaN,0.0,0.0
2023-11-23 21:10:00+00:00,0.0,0.0,0.0,0.0,NaN,0.0,0.0
2023-11-23 21:15:00+00:00,0.0,0.0,0.0,0.0,NaN,0.0,0.0
2023-11-23 21:20:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2023-11-23 21:25:00+00:00,1.0,0.0,0.0,0.0,0.0,0.2,1.0


In [6]:
for minutes in [20, 25]:
    past_times = feature_data.index - pd.Timedelta(minutes=minutes)

    feature_data[f"lag_{minutes}min"] = (
        data["current_occupancy"]
        .reindex(past_times)
        .to_numpy()
    )

feature_data[
    ["current_occupancy",
     "lag_5min", "lag_10min", "lag_15min",
     "lag_20min", "lag_25min", "lag_30min",
     "target_30min"]
].head(10)

,current_occupancy,lag_5min,lag_10min,lag_15min,lag_20min,lag_25min,lag_30min,target_30min
collecteddate,,,,,,,,
2023-11-23 19:15:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2023-11-23 20:50:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2023-11-23 20:55:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0
2023-11-23 21:00:00+00:00,0.0,0.0,0.0,NaN,NaN,NaN,NaN,1.0
2023-11-23 21:05:00+00:00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0
2023-11-23 21:10:00+00:00,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0
2023-11-23 21:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0
2023-11-23 21:20:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2023-11-23 21:25:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
history_columns = [
    "current_occupancy",
    "lag_5min",
    "lag_10min",
    "lag_15min",
    "lag_20min",
    "lag_25min",
    "lag_30min"
]

feature_data["recent_occupancy_rate"] = (
    feature_data[history_columns].mean(axis=1)
)

feature_data[
    history_columns +
    ["recent_occupancy_rate", "target_30min"]
].head(10)

,current_occupancy,lag_5min,lag_10min,lag_15min,lag_20min,lag_25min,lag_30min,recent_occupancy_rate,target_30min
collecteddate,,,,,,,,,
2023-11-23 19:15:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.0
2023-11-23 20:50:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.0
2023-11-23 20:55:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.000000,1.0
2023-11-23 21:00:00+00:00,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.000000,1.0
2023-11-23 21:05:00+00:00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.000000,0.0
2023-11-23 21:10:00+00:00,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.000000,0.0
2023-11-23 21:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.000000,0.0
2023-11-23 21:20:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.0
2023-11-23 21:25:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.142857,1.0


In [8]:
required_columns = [
    "current_occupancy",
    "lag_5min",
    "lag_10min",
    "lag_15min",
    "lag_20min",
    "lag_25min",
    "lag_30min",
    "hour",
    "day_of_week",
    "target_30min"
]

final_data = feature_data.dropna(
    subset=required_columns
).copy()

# Recalculate the recent occupancy rate using only complete histories
final_data["recent_occupancy_rate"] = (
    final_data[
        ["current_occupancy",
         "lag_5min", "lag_10min", "lag_15min",
         "lag_20min", "lag_25min", "lag_30min"]
    ].mean(axis=1)
)

print("Samples before removing incomplete histories:", len(feature_data))
print("Samples after removing incomplete histories:", len(final_data))

print("\nMissing values in final dataset:")
print(final_data.isna().sum())

final_data.head(10)

Samples before removing incomplete histories: 127428
Samples after removing incomplete histories: 107059

Missing values in final dataset:
current_occupancy        0
target_30min             0
lag_5min                 0
lag_10min                0
lag_15min                0
lag_30min                0
hour                     0
day_of_week              0
recent_occupancy_rate    0
lag_20min                0
lag_25min                0
dtype: int64


,current_occupancy,target_30min,lag_5min,lag_10min,lag_15min,lag_30min,hour,day_of_week,recent_occupancy_rate,lag_20min,lag_25min
collecteddate,,,,,,,,,,,
2023-11-23 21:20:00+00:00,0.0,1.0,0.0,0.0,0.0,0.0,21,3,0.000000,0.0,0.0
2023-11-23 21:25:00+00:00,1.0,1.0,0.0,0.0,0.0,0.0,21,3,0.142857,0.0,0.0
2023-11-23 21:30:00+00:00,1.0,0.0,1.0,0.0,0.0,0.0,21,3,0.285714,0.0,0.0
2023-11-23 21:35:00+00:00,0.0,0.0,1.0,1.0,0.0,0.0,21,3,0.285714,0.0,0.0
2023-11-23 21:40:00+00:00,0.0,0.0,0.0,1.0,1.0,0.0,21,3,0.285714,0.0,0.0
2023-11-23 22:55:00+00:00,1.0,1.0,1.0,0.0,1.0,0.0,22,3,0.428571,0.0,0.0
2023-11-23 23:00:00+00:00,1.0,1.0,1.0,1.0,0.0,0.0,23,3,0.571429,1.0,0.0
2023-11-23 23:05:00+00:00,1.0,0.0,1.0,1.0,1.0,0.0,23,3,0.714286,0.0,1.0
2023-11-23 23:10:00+00:00,1.0,0.0,1.0,1.0,1.0,1.0,23,3,0.857143,1.0,0.0


In [9]:
print("Final dataset shape:", final_data.shape)

print("\nTarget distribution:")
print(final_data["target_30min"].value_counts())

print("\nTarget percentage:")
print(
    final_data["target_30min"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

Final dataset shape: (107059, 11)

Target distribution:
target_30min
1.0    71785
0.0    35274
Name: count, dtype: int64

Target percentage:
target_30min
0.0    32.948187
1.0    67.051813
Name: proportion, dtype: float64


In [10]:
feature_file = "/content/drive/MyDrive/MMA3001_Project/data/processed/occupancy_features.csv"

final_data.to_csv(feature_file, index=True)

print("Feature dataset saved to:")
print(feature_file)

print("\nSaved shape:")
print(final_data.shape)

Feature dataset saved to:
/content/drive/MyDrive/MMA3001_Project/data/processed/occupancy_features.csv

Saved shape:
(107059, 11)
